# Exploratory Data Analysis (EDA)
## Wine Quality Prediction Using Decision Tree, Random Forest, and Gradient Boosting

This notebook performs Exploratory Data Analysis on the **Wine Quality Dataset** used for the project.

### Project objective
Analyze the physicochemical properties of wine and the target **quality** score before building the Decision Tree, Random Forest, and Gradient Boosting models.

### Features
- Fixed acidity
- Volatile acidity
- Citric acid
- Residual sugar
- Chlorides
- Free sulfur dioxide
- Total sulfur dioxide
- Density
- pH
- Sulphates
- Alcohol

**Target:** `quality`


In [ ]:
# 1. Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")


In [ ]:
# 2. Load the Wine Quality dataset
# In Google Colab, upload the CSV first or change DATA_PATH to your file location.

DATA_PATH = "../data/winequality-red.csv"

# The common UCI/Kaggle wine-quality CSV uses ';' as the separator.
# sep=None with the Python engine can also detect comma-separated CSV files.
df = pd.read_csv(DATA_PATH, sep=None, engine="python")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


In [ ]:
# 3. Basic dataset information
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()


In [ ]:
# 4. Check for duplicate records
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    print("Duplicate rows are present and should be considered during preprocessing.")
else:
    print("No duplicate rows were found.")


In [ ]:
# 5. Check for missing values
missing_values = df.isnull().sum()

print("Missing values by column:")
display(missing_values.to_frame("Missing Values"))

plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Value Heatmap")
plt.xlabel("Features")
plt.ylabel("Records")
plt.tight_layout()
plt.show()


### Observation — Missing Values

The missing-value table and heatmap show whether any physicochemical feature contains null values. If all columns contain zero missing values, no missing-value imputation is required at this stage.


In [ ]:
# 6. Separate numerical and categorical features
numerical_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


In [ ]:
# 7. Descriptive statistics
print("Descriptive Statistics:")
display(df.describe().T)


In [ ]:
# 8. Distribution of the target variable: Wine Quality
if "quality" not in df.columns:
    raise ValueError("Target column 'quality' was not found. Check the dataset column name.")

plt.figure(figsize=(9, 5))
sns.countplot(data=df, x="quality")
plt.title("Distribution of Wine Quality")
plt.xlabel("Wine Quality Score")
plt.ylabel("Number of Wines")
plt.tight_layout()
plt.show()

print("Quality value counts:")
display(df["quality"].value_counts().sort_index())


### Observation — Wine Quality Distribution

The target distribution shows how many wine samples belong to each quality score. This is important before model training because an uneven distribution of quality classes can affect classification performance and the interpretation of accuracy, precision, recall, and F1-score.


In [ ]:
# 9. Histograms of physicochemical features
feature_columns = [col for col in df.columns if col != "quality"]

df[feature_columns].hist(bins=20, figsize=(16, 12), layout=(4, 3))
plt.suptitle("Distributions of Wine Physicochemical Features", fontsize=16, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


### Observation — Feature Distributions

The histograms help identify the distribution, spread, skewness, and possible unusual values of the physicochemical variables. These patterns are useful when deciding what preprocessing is required before applying the machine-learning models.


In [ ]:
# 10. Correlation matrix
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
corr_matrix = df[numerical_cols].corr()

print("--- Correlation Matrix ---")
display(corr_matrix.round(2))

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title("Correlation Heatmap of Wine Quality Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


### Observation — Correlation Heatmap

The correlation heatmap is used to understand the linear relationships among the physicochemical features and the `quality` target.

- A value close to **+1** indicates a strong positive relationship.
- A value close to **0** indicates a weak or negligible linear relationship.
- A value close to **-1** indicates a strong negative relationship.

The correlations should be interpreted as relationships in this dataset, not as proof that one variable causes another.


In [ ]:
# 11. Correlation of each feature with wine quality
quality_corr = corr_matrix["quality"].sort_values(ascending=False)

print("Correlation of features with wine quality:")
display(quality_corr.to_frame("Correlation with Quality"))

plt.figure(figsize=(9, 6))
quality_corr.drop("quality").sort_values().plot(kind="barh")
plt.title("Feature Correlation with Wine Quality")
plt.xlabel("Pearson Correlation")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# 12. Boxplots: numerical features vs Wine Quality
# Boxplots help identify how feature values vary across quality scores
for col in feature_columns:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x="quality", y=col)
    plt.title(f"{col} vs Wine Quality")
    plt.xlabel("Wine Quality Score")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


### Observation — Boxplots

The boxplots compare each physicochemical feature across wine-quality scores. They can reveal differences in the distributions of features between quality classes and highlight potential outliers.

These observations can guide the preprocessing and model-development stages.


In [ ]:
# 13. Selected feature relationships with quality
# These plots give a compact visual view of important feature-target relationships.

selected_features = ["alcohol", "volatile acidity", "sulphates", "citric acid"]

available_features = [col for col in selected_features if col in df.columns]

for col in available_features:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="quality", y=col)
    plt.title(f"{col} vs Wine Quality")
    plt.xlabel("Wine Quality")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


In [ ]:
# 14. Pairplot for selected variables
pairplot_features = [col for col in ["alcohol", "volatile acidity", "sulphates", "citric acid", "quality"] if col in df.columns]

if len(pairplot_features) >= 2:
    sns.pairplot(
        df[pairplot_features],
        hue="quality",
        corner=True
    )
    plt.suptitle("Pairwise Relationships of Selected Wine Features", y=1.02)
    plt.show()


In [ ]:
# 15. Outlier overview using the IQR method
outlier_summary = []

for col in feature_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary.append({
        "Feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Outlier Count": int(count)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values("Outlier Count", ascending=False)

print("Potential outliers by feature using the 1.5 × IQR rule:")
display(outlier_df)


In [ ]:
# 16. Save the main EDA figures/data for the project report
output_dir = "../outputs/eda"
os.makedirs(output_dir, exist_ok=True)

# Save correlation matrix
corr_matrix.to_csv(os.path.join(output_dir, "wine_correlation_matrix.csv"))

# Save feature-quality correlations
quality_corr.to_csv(os.path.join(output_dir, "quality_feature_correlations.csv"))

# Save outlier summary
outlier_df.to_csv(os.path.join(output_dir, "wine_outlier_summary.csv"), index=False)

print("EDA output files saved to:", output_dir)


In [ ]:
# 17. Export a compact EDA report as PDF
from matplotlib.backends.backend_pdf import PdfPages

pdf_path = os.path.join(output_dir, "Wine_Quality_EDA_Report.pdf")

with PdfPages(pdf_path) as pdf:
    # Target distribution
    fig = plt.figure(figsize=(9, 5))
    sns.countplot(data=df, x="quality")
    plt.title("Distribution of Wine Quality")
    plt.xlabel("Wine Quality Score")
    plt.ylabel("Number of Wines")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    # Correlation heatmap
    fig = plt.figure(figsize=(12, 9))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
    plt.title("Correlation Heatmap of Wine Quality Dataset")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    # Quality correlations
    fig = plt.figure(figsize=(9, 6))
    quality_corr.drop("quality").sort_values().plot(kind="barh")
    plt.title("Feature Correlation with Wine Quality")
    plt.xlabel("Pearson Correlation")
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

print("EDA PDF report saved to:", pdf_path)


## EDA Conclusion

The EDA stage provides an understanding of the Wine Quality Dataset before model development. It examines:

1. Dataset structure and data types
2. Duplicate and missing values
3. Descriptive statistics
4. Wine-quality class distribution
5. Feature distributions
6. Feature-to-feature and feature-to-quality correlations
7. Feature behavior across quality scores
8. Potential outliers

The cleaned and understood dataset can then be used for the project's supervised learning stage with **Decision Tree, Random Forest, and Gradient Boosting**, followed by evaluation using **accuracy, precision, recall, and F1-score**.
